# Data Loading and Preprocessing

## 1. Match ROI file paths to respective DICOM files

In [ ]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT = Path("./INbreast/")

In [3]:
inbreast_df = pd.read_csv(f"{ROOT}/INbreast_cleaned.csv")

Get and store file paths to respective .roi files

In [4]:
def get_dicom_file_path(file_name):
    dicom_root = Path(f"{ROOT}/AllDICOMs/")
    matches = list(dicom_root.rglob(f"{file_name}*"))
    if not matches:
        raise FileNotFoundError(f"No DICOM file found for {file_name}")
    return matches[0].as_posix()

def get_roi_file_path(file_name):
    return Path(f"{ROOT}/AllXML/{file_name}.xml").as_posix()

inbreast_df["dicom_path_dir"] = inbreast_df["File Name"].apply(get_dicom_file_path)
inbreast_df["roi_path_dir"] = inbreast_df["File Name"].apply(get_roi_file_path)

Remove all rows containing any null value in any column from the dataframe

In [5]:
inbreast_df = inbreast_df.dropna().reset_index(drop=True)

Save updated metadata file

In [6]:
inbreast_df.to_csv(f"{ROOT}/new_INbreast_cleaned.csv", index=False)

## 2. Read DICOM files

In [7]:
import pydicom
import numpy as np
import cv2

Function to read DICOM image

In [8]:
def load_dicom_image(dicom_path):
    ds = pydicom.dcmread(dicom_path)
    img = ds.pixel_array.astype(np.float32)

    if ds.PhotometricInterpretation == "MONOCHROME1":
        img = img.max() - img

    return img

## 3. Convert ROI points to full-image bounding box coordinates

Function to get bounding box from ROI XML

In [9]:
import plistlib
import re

def parse_inbreast_roi(xml_path):
    with open(xml_path, 'rb') as f:
        plist = plistlib.load(f)

    bboxes = []
    for image in plist.get('Images', []):
        for roi in image.get('ROIs', []):
            points_px = roi.get('Point_px', [])
            if not points_px:
                continue

            coords = []
            for pt_str in points_px:
                match = re.match(r'\(([-\d.]+),\s*([-\d.]+)\)', pt_str)
                if match:
                    coords.append((float(match.group(1)), float(match.group(2))))

            if not coords:
                continue

            xs = [c[0] for c in coords]
            ys = [c[1] for c in coords]
            bboxes.append([min(xs), min(ys), max(xs), max(ys)])
            
    return bboxes

## 4. Percentile intensity clipping

Function to perform percentile clipping

In [10]:
def percentile_clip(img, lower=1, upper=99):
    p_low = np.percentile(img, lower)
    p_high = np.percentile(img, upper)

    img = np.clip(img, p_low, p_high)
    
    return img

## 5. Normalise pixel values

Function to normalise pixel values to 0-1

In [11]:
def normalize_0_to_1(img):
    img_min = img.min()
    img_max = img.max()

    img = (img - img_min) / (img_max - img_min + 1e-6)

    return img.astype(np.float32)

## 6. Replicate grayscale image into three channels

Function to convert grayscale to 3 channels

In [12]:
def to_three_channels(img):
    return np.stack([img, img, img], axis=-1)

## 7. Letterbox resize

Function to perform letterbox resize on full image and adjust all bounding boxes

In [13]:
def letterbox(img, bboxes, new_size=1024):
    h, w = img.shape[:2]

    scale = min(new_size / w, new_size / h)

    resized_w = int(w * scale)
    resized_h = int(h * scale)

    resized_img = cv2.resize(img, (resized_w, resized_h))

    canvas = np.zeros((new_size, new_size, 3), dtype=np.float32)

    pad_x = (new_size - resized_w) // 2
    pad_y = (new_size - resized_h) // 2

    canvas[pad_y:pad_y + resized_h, pad_x:pad_x + resized_w] = resized_img

    # Update all bounding boxes after resizing
    new_bboxes = []
    for bbox in bboxes:
        x_min, y_min, x_max, y_max = bbox
        new_bboxes.append([
            x_min * scale + pad_x,
            y_min * scale + pad_y,
            x_max * scale + pad_x,
            y_max * scale + pad_y
        ])

    return canvas, new_bboxes

## 8. Convert to bounding box coordinates YOLO format

Create class mapping to get class ID

In [14]:
CLASS_MAP_2c = {
    "1": 0,
    "2": 0,
    "3": 0,
    "4a": 1,
    "4b": 1,
    "4c": 1,
    "5": 1,
    "6": 1,
}

Function to get class ID

In [15]:
def get_class_id(row, class_map):
    bi_rads = row["Bi-Rads"].strip()

    if bi_rads not in class_map:
        raise ValueError(f"Unknown class: {bi_rads}")

    return class_map[bi_rads]

Function to convert bounding box to YOLO format

In [16]:
def bbox_to_yolo(class_id, bbox, img_size=1024):
    x_min, y_min, x_max, y_max = bbox

    x_center = ((x_min + x_max) / 2) / img_size
    y_center = ((y_min + y_max) / 2) / img_size
    width = (x_max - x_min) / img_size
    height = (y_max - y_min) / img_size

    return [class_id, x_center, y_center, width, height]

Function to save processed image and YOLO label

In [17]:
def save_processed_sample(img, yolo_labels, image_output_path, label_output_path):
    # Convert 0–1 image to 0–255 PNG
    img_uint8 = (img * 255).astype(np.uint8)

    cv2.imwrite(str(image_output_path), img_uint8)

    with open(label_output_path, "w") as f:
        for yolo_label in yolo_labels:
            f.write(
                f"{int(yolo_label[0])} "
                f"{yolo_label[1]:.6f} "
                f"{yolo_label[2]:.6f} "
                f"{yolo_label[3]:.6f} "
                f"{yolo_label[4]:.6f}\n"
            )

## 8. Full preprocessing

Function to process one full mammogram with multiple lesions

In [18]:
def preprocess_inbreast_row(
    row,
    class_map,
    image_output_path,
    label_output_path,
    img_size=1024,
):
    # Load full mammogram
    full_image_path = row["dicom_path_dir"]
    full_img = load_dicom_image(full_image_path)

    # Get bounding boxes from INbreast XML
    xml_path = Path(row["roi_path_dir"])

    # Skip mammograms with no lesion annotation
    if not xml_path.exists():
        print(f"No lesion annotation: {full_image_path}")
        return False

    roi_bboxes = parse_inbreast_roi(xml_path)

    bboxes = []
    for bbox in roi_bboxes:
        x_min, y_min, x_max, y_max = bbox

        bboxes.append([
            x_min,
            y_min,
            x_max,
            y_max
        ])

    # Skip if XML exists but contains no valid lesion boxes
    if len(roi_bboxes) == 0:
        print(f"No valid boxes: {full_image_path}")
        return False

    # Get lesion class
    class_id = get_class_id(row, class_map)

    # Percentile intensity clipping
    full_img = percentile_clip(full_img)

    # Normalise pixel values
    full_img = normalize_0_to_1(full_img)

    # Replicate grayscale image into three channels
    full_img = to_three_channels(full_img)

    # Letterbox resize
    full_img, bboxes = letterbox(
        full_img,
        bboxes,
        new_size=img_size
    )

    # Convert bounding box coordinates to YOLO format
    yolo_labels = [
        bbox_to_yolo(class_id, bbox, img_size=img_size)
        for bbox in bboxes
    ]

    for label in yolo_labels:
        _, x, y, w, h = label

        if not (
            0 <= x <= 1
            and 0 <= y <= 1
            and 0 <= w <= 1
            and 0 <= h <= 1
        ):
            raise ValueError(f"Invalid YOLO label: {label}")

    # Save processed image and label
    save_processed_sample(
        full_img,
        yolo_labels,
        image_output_path,
        label_output_path
    )

    return True

Function to process CSV file

In [19]:
def process_metadata_csv(
    metadata_csv,
    class_map,
    image_output_dir,
    label_output_dir,
    img_size=1024
):
    df = pd.read_csv(metadata_csv)

    image_output_dir = Path(image_output_dir)
    label_output_dir = Path(label_output_dir)

    image_output_dir.mkdir(parents=True, exist_ok=True)
    label_output_dir.mkdir(parents=True, exist_ok=True)

    total = len(df)

    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        file_name = row["File Name"]
        
        try:
            image_output_path = image_output_dir / f"{file_name}.png"
            label_output_path = label_output_dir / f"{file_name}.txt"

            processed = preprocess_inbreast_row(
                row=row,
                class_map=class_map,
                image_output_path=image_output_path,
                label_output_path=label_output_path,
                img_size=img_size
            )

            if processed:
                print(f"[{idx}/{total}] Processed: {file_name}")
            else:
                print(f"[{idx}/{total}] Skipped: {file_name}")

        except Exception as e:
            print(f"[{idx}/{total}] Failed: {file_name}")
            print(e)

Execute preprocessing

In [20]:
OUT_DIR = Path("./inbreast_YOLO_labelled_2c")
class_map = CLASS_MAP_2c

In [21]:
csv_file = f"{ROOT}/new_INbreast_cleaned.csv"

process_metadata_csv(
    metadata_csv=csv_file,
    class_map=class_map,
    image_output_dir=f"{OUT_DIR}/images/",
    label_output_dir=f"{OUT_DIR}/labels/",
    img_size=1024
)

No lesion annotation: INbreast/AllDICOMs/22678622_61b13c59bcba149e_MG_R_CC_ANON.dcm
[1/410] Skipped: 22678622
[2/410] Processed: 22678646
No lesion annotation: INbreast/AllDICOMs/22678670_61b13c59bcba149e_MG_R_ML_ANON.dcm
[3/410] Skipped: 22678670
[4/410] Processed: 22678694
[5/410] Processed: 22614074
[6/410] Processed: 22614097
[7/410] Processed: 22614127
[8/410] Processed: 22614150
[9/410] Processed: 50997434
[10/410] Processed: 50997461
[11/410] Processed: 50997488
[12/410] Processed: 50997515
[13/410] Processed: 24055445
[14/410] Processed: 24055464
[15/410] Processed: 24055483
[16/410] Processed: 24055502
[17/410] Processed: 22580192
No lesion annotation: INbreast/AllDICOMs/22580218_5530d5782fc89dd7_MG_L_CC_ANON.dcm
[18/410] Skipped: 22580218
[19/410] Processed: 22580244
No lesion annotation: INbreast/AllDICOMs/22580270_5530d5782fc89dd7_MG_L_ML_ANON.dcm
[20/410] Skipped: 22580270
[21/410] Processed: 50998032
[22/410] Processed: 50998059
[23/410] Processed: 50998086
[24/410] Proce